Scrapping From Inaturalist

In [ ]:
BASE_DIR = r"Klasifikasi Kutu\Input2"
TARGET_PER_SPECIES = 1000
THREADS = 15

API_URL = "https://api.inaturalist.org/v1/observations"

species_list = [
    "Dermacentor variabilis",
    "Ixodes scapularis",
    "Rhipicephalus sanguineus"
]


def validate_image(img_bytes):
    try:
        img = Image.open(BytesIO(img_bytes))
        img.verify()
        return True
    except:
        return False


def download_one(url, save_path):
    for _ in range(3):  # retry 3x
        try:
            r = requests.get(url, timeout=10)
            if r.status_code == 200 and validate_image(r.content):
                with open(save_path, "wb") as f:
                    f.write(r.content)
                return True
        except:
            time.sleep(1)
    return False


def scrape_species(species):
    print(f"\n=== SCRAPING: {species} ===")

    folder = os.path.join(BASE_DIR, species)
    os.makedirs(folder, exist_ok=True)

    collected_urls = []

    # == 1. Ambil total data dulu ==
    params = {
        "taxon_name": species,
        "photos": True,
        "per_page": 1,
    }
    resp = requests.get(API_URL, params=params).json()
    total_results = resp.get("total_results", 0)
    if total_results == 0:
        print("[ERROR] Tidak ada data!")
        return
    
    per_page = 200  # MAX
    total_pages = math.ceil(total_results / per_page)

    print(f"[INFO] Total hasil: {total_results}, Total halaman: {total_pages}")

    # == 2. Loop semua halaman untuk kumpulkan URL ==
    for page in range(1, total_pages + 1):
        params = {
            "taxon_name": species,
            "photos": True,
            "page": page,
            "per_page": per_page,
        }

        data = requests.get(API_URL, params=params).json().get("results", [])

        for obs in data:
            if not obs.get("photos"):
                continue
            url = obs["photos"][0]["url"].replace("square", "large")
            collected_urls.append(url)

        if len(collected_urls) >= TARGET_PER_SPECIES:
            break

    print(f"[INFO] Total URL terkumpul: {len(collected_urls)}")

    # == 3. Download dengan multithread ==
    tasks = []
    with ThreadPoolExecutor(max_workers=THREADS) as exe:
        for i, url in enumerate(collected_urls[:TARGET_PER_SPECIES]):
            ext = url.split(".")[-1].lower()
            if ext not in ["jpg", "jpeg", "png"]:
                ext = "jpg"
            save_path = os.path.join(folder, f"{i+1}.{ext}")
            tasks.append(exe.submit(download_one, url, save_path))

        for _ in tqdm(tasks, desc=f"Mengunduh {species}"):
            _.result()

    print(f"[DONE] Gambar tersimpan: {min(TARGET_PER_SPECIES, len(collected_urls))}")
for sp in species_list:
    scrape_species(sp)

In [ ]:
base_folder = r"Input"

# ekstensi gambar
extensions = ["*.jpg", "*.jpeg", "*.png", "*.bmp", "*.gif"]

# loop setiap subfolder
for subfolder in os.listdir(base_folder):
    folder_path = os.path.join(base_folder, subfolder)

    if os.path.isdir(folder_path):
        images = []
        for ext in extensions:
            images.extend(glob.glob(os.path.join(folder_path, ext)))

        print(f"{subfolder}: {len(images)} gambar")


Scrap from E-Tick

In [ ]:
import requests

url = "https://www.etick.ca/api/publicmetas"

params = {
    "lang": "en",
    "limit": 20,
    "page": 1,
    "species_id": 31
}

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.etick.ca/"
}

r = requests.get(url, params=params, headers=headers)

print(r.status_code)
print(r.headers.get("Content-Type"))
print(r.json())

In [ ]:
import requests
import pandas as pd
import os
import time

# =========================
# CONFIG
# =========================

BASE_URL = "https://www.etick.ca/api/publicmetas"
BASE_IMG_URL = "https://www.etick.ca/attachments/observations/square/"

HEADERS = {
    "User-Agent": "Mozilla/5.0",
    "Accept": "application/json, text/plain, */*",
    "Referer": "https://www.etick.ca/"
}

SPECIES = {
    5: "Dermacentor_variabilis",
    13: "Ixodes_scapularis",
    31: "Rhipicephalus_sanguineus",
}

TOTAL_PER_SPECIES = 500


# =========================
# FUNCTION SCRAPE PER SPECIES
# =========================

def scrape_species(species_id, species_name):

    print(f"\nScraping {species_name}...")

    all_data = []
    page = 1

    folder_path = os.path.join("Input2", species_name)
    os.makedirs(folder_path, exist_ok=True)

    while len(all_data) < TOTAL_PER_SPECIES:

        params = {
            "lang": "en",
            "limit": 100,
            "page": page,
            "species_id": species_id
        }

        response = requests.get(BASE_URL, headers=HEADERS, params=params)

        if response.status_code != 200:
            print(f"[ERROR] Gagal mengambil data di page {page}")
            break

        data = response.json()
        items = data.get("data", [])

        if not items:
            print("Tidak ada data lagi.")
            break

        for item in items:

            observation_id = item.get("observation_id")

            for p in item.get("photo", []):

                photo_path = p.get("photo_path")
                photo_id = p.get("photo_id")

                if not photo_path:
                    continue

                img_url = BASE_IMG_URL + photo_path
                filename = f"{observation_id}_{photo_id}.jpg"
                save_path = os.path.join(folder_path, filename)

                if os.path.exists(save_path):
                    continue

                try:
                    r = requests.get(img_url, headers=HEADERS, timeout=10)

                    if r.status_code == 200:
                        with open(save_path, "wb") as f:
                            f.write(r.content)

                        print("Downloaded:", filename)
                    else:
                        print("Failed:", img_url)

                except Exception as e:
                    print("Download error:", e)

                time.sleep(0.2)  # polite delay

        all_data.extend(items)

        print(f"Page {page} done")
        page += 1
        time.sleep(1)

    print(f"{species_name} selesai. Total observation: {len(all_data)}")


# =========================
# MAIN LOOP
# =========================

for species_id, species_name in SPECIES.items():
    scrape_species(species_id, species_name)

print("\nSemua selesai 🚀")
